# Library Management System


- Patterns
    - decorator
    - strategy
    - command
    - observer
    - singleton
    - factory
    - abstract factory
    - adapter

In [ ]:
# Singleton Pattern
class Library:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(Library, cls).__new__(cls)
            cls._instance.books = []
            cls._instance.borrowed_books = []
            cls._instance.members = []
        return cls._instance

    def borrow_book(self, book):
        if book in self.books:
            self.books.remove(book)
            self.borrowed_books.append(book)
            return True
        return False

    def return_book(self, book):
        if book in self.borrowed_books:
            self.borrowed_books.remove(book)
            self.books.append(book)
            return True
        return False

In [ ]:
from abc import ABC, abstractmethod

# Factory Method Pattern
class BookFactory:
    @staticmethod
    def create_book(book_type, title, author):
        if book_type == "EBook":
            return EBook(title, author)
        elif book_type == "PrintBook":
            return PrintBook(title, author)
        else:
            return None

In [ ]:
# Abstract Factory Pattern
class MemberFactory(ABC):
    @abstractmethod
    def create_member(self, name):
        pass

class StudentMemberFactory(MemberFactory):
    def create_member(self, name):
        return StudentMember(name)

class FacultyMemberFactory(MemberFactory):
    def create_member(self, name):
        return FacultyMember(name)

In [ ]:
# Decorator Pattern
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def get_description(self):
        return f"{self.title} by {self.author}"

class MetadataDecorator(Book):
    def __init__(self, book, metadata):
        self._book = book
        self._metadata = metadata

    def get_description(self):
        return f"{self._book.get_description()} with metadata: {self._metadata}"

class EBook(Book):
    pass

class PrintBook(Book):
    pass

In [ ]:
# Observer Pattern
class Notifier:
    def __init__(self):
        self._observers = []
        self.new_book = None

    def attach(self, observer):
        self._observers.append(observer)

    def detach(self, observer):
        self._observers.remove(observer)

    def notify(self):
        for observer in self._observers:
            observer.update(self)

class LibraryNotifier(Notifier):
    def __init__(self):
        super().__init__()

    def add_new_book(self, book):
        self.new_book = book
        self.notify()

class Member(ABC):
    @abstractmethod
    def update(self, notifier):
        pass

class StudentMember(Member):
    def __init__(self, name):
        self.name = name

    def update(self, notifier):
        print(f"Student {self.name} notified of new book: {notifier.new_book.get_description()}")

class FacultyMember(Member):
    def __init__(self, name):
        self.name = name

    def update(self, notifier):
        print(f"Faculty {self.name} notified of new book: {notifier.new_book.get_description()}")

In [ ]:
# Strategy Pattern
class SearchStrategy(ABC):
    @abstractmethod
    def search(self, library, query):
        pass

class TitleSearchStrategy(SearchStrategy):
    def search(self, library, query):
        available_books = [book for book in library.books if query.lower() in book.title.lower()]
        borrowed_books = [book for book in library.borrowed_books if query.lower() in book.title.lower()]
        return available_books, borrowed_books

class AuthorSearchStrategy(SearchStrategy):
    def search(self, library, query):
        available_books = [book for book in library.books if query.lower() in book.author.lower()]
        borrowed_books = [book for book in library.borrowed_books if query.lower() in book.author.lower()]
        return available_books, borrowed_books

In [ ]:
# Command Pattern
class Command(ABC):
    @abstractmethod
    def execute(self):
        pass

class BorrowBookCommand(Command):
    def __init__(self, library, title):
        self.library = library
        self.title = title
        self.result = ""

    def execute(self):
        book = next((book for book in self.library.books if book.title == self.title), None)
        if book and self.library.borrow_book(book):
            self.result = f"Borrowed book: {book.get_description()}"
        else:
            self.result = f"Book '{self.title}' not found or already borrowed"

class ReturnBookCommand(Command):
    def __init__(self, library, title):
        self.library = library
        self.title = title
        self.result = ""

    def execute(self):
        book = next((book for book in self.library.borrowed_books if book.title == self.title), None)
        if book and self.library.return_book(book):
            self.result = f"Returned book: {book.get_description()}"
        else:
            self.result = f"Book '{self.title}' not found in borrowed books"


In [ ]:
# Adapter Pattern
class ThirdPartyBookAPI:
    def get_books(self):
        return [{"title": "ThirdPartyBook1", "author": "Author1"}, {"title": "ThirdPartyBook2", "author": "Author2"}]

class BookAdapter(Book):
    def __init__(self, third_party_book):
        super().__init__(third_party_book["title"], third_party_book["author"])


In [ ]:
import tkinter as tk
from tkinter import messagebox

# Config
SUPPORTED_BOOK_TYPES = ["EBook", "PrintBook"]
SEARCH_STRATEGIES = {
    "Search by Title": TitleSearchStrategy,
    "Search by Author": AuthorSearchStrategy
}


# Client Class using Tkinter
class Client(tk.Tk):
    def __init__(self):
        super().__init__()
        
        self.search_strategies = SEARCH_STRATEGIES
        self.library = Library()
        self.notifier = LibraryNotifier()

        self.title("Library Management System")
        self.geometry("")

        self.create_widgets()

        # Configure grid layout to expand with window size
        self.grid_columnconfigure(1, weight=1)
        self.grid_columnconfigure(2, weight=1)
        self.grid_columnconfigure(3, weight=1)
        self.grid_columnconfigure(4, weight=1)
        self.grid_columnconfigure(5, weight=1)
        self.grid_rowconfigure(0, weight=1)
        self.grid_rowconfigure(1, weight=1)
        self.grid_rowconfigure(2, weight=1)

        # Initialize and register members
        self.init_members()

    def init_members(self):
        student_factory = StudentMemberFactory()
        faculty_factory = FacultyMemberFactory()

        student = student_factory.create_member("Alice")
        faculty = faculty_factory.create_member("Dr. Smith")

        self.library.members.append(student)
        self.library.members.append(faculty)

        self.notifier.attach(student)
        self.notifier.attach(faculty)

    def create_widgets(self):
        # Add book section
        tk.Label(self, text="Add Book").grid(row=0, column=0, padx=10, pady=10, sticky="e")
        
        # Dynamic selector for book type
        self.book_type_var = tk.StringVar(value=SUPPORTED_BOOK_TYPES[0])
        book_type_selector = tk.OptionMenu(self, self.book_type_var, *SUPPORTED_BOOK_TYPES)
        book_type_selector.grid(row=0, column=1, padx=10, pady=10, sticky="ew")
        
        self.title_var = tk.StringVar()
        self.author_var = tk.StringVar()

        tk.Label(self, text="Title:").grid(row=0, column=2, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.title_var).grid(row=0, column=3, padx=10, pady=10, sticky="ew")
        
        tk.Label(self, text="Author:").grid(row=0, column=4, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.author_var).grid(row=0, column=5, padx=10, pady=10, sticky="ew")
        
        tk.Button(self, text="Add", command=self.add_book).grid(row=0, column=6, padx=10, pady=10, sticky="ew")

        # Button to add third-party books
        tk.Button(self, text="Add Third-Party Books", command=self.add_third_party_books).grid(row=0, column=7, padx=10, pady=10, sticky="ew")

        # Search book section
        tk.Label(self, text="Search Book").grid(row=1, column=0, padx=10, pady=10, sticky="e")
        self.search_var = tk.StringVar()
        self.search_result_var = tk.StringVar()

        tk.Label(self, text="Query:").grid(row=1, column=1, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.search_var).grid(row=1, column=2, padx=10, pady=10, sticky="ew")
        
        # Dynamically create search buttons based on available strategies
        col = 3
        for strategy_name, strategy_class in self.search_strategies.items():
            tk.Button(self, text=strategy_name, command=lambda s=strategy_class: self.search_books(s)).grid(row=1, column=col, padx=10, pady=10, sticky="ew")
            col += 1
        
        tk.Label(self, textvariable=self.search_result_var).grid(row=1, column=col, padx=10, pady=10, sticky="ew")

        # Borrow and Return book section
        tk.Label(self, text="Borrow/Return Book").grid(row=2, column=0, padx=10, pady=10, sticky="e")
        self.borrow_return_var = tk.StringVar()

        tk.Label(self, text="Book Title:").grid(row=2, column=1, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.borrow_return_var).grid(row=2, column=2, padx=10, pady=10, sticky="ew")
        tk.Button(self, text="Borrow", command=self.borrow_book).grid(row=2, column=3, padx=10, pady=10, sticky="ew")
        tk.Button(self, text="Return", command=self.return_book).grid(row=2, column=4, padx=10, pady=10, sticky="ew")


    def add_book(self):
        book_type = self.book_type_var.get()
        title = self.title_var.get()
        author = self.author_var.get()
        book = BookFactory.create_book(book_type, title, author)
        if book:
            self.library.books.append(book)
            self.notifier.add_new_book(book)
            messagebox.showinfo("Success", f"Added {book.get_description()}")
        else:
            messagebox.showerror("Error", "Failed to add book")

    def search_books(self, strategy):
        query = self.search_var.get()
        available_books, borrowed_books = strategy().search(self.library, query)
        self.display_search_results(available_books, borrowed_books)

    def display_search_results(self, available_books, borrowed_books):
        if available_books or borrowed_books:
            result_text = ""
            if available_books:
                result_text += "Available books:\n"
                result_text += "\n".join([book.get_description() for book in available_books]) + "\n"
            if borrowed_books:
                result_text += "Borrowed books:\n"
                result_text += "\n".join([book.get_description() for book in borrowed_books])
            self.search_result_var.set(result_text)
        else:
            self.search_result_var.set("No results found")
        messagebox.showinfo("Search Results", self.search_result_var.get())

    def borrow_book(self):
        title = self.borrow_return_var.get()
        command = BorrowBookCommand(self.library, title)
        command.execute()
        messagebox.showinfo("Result", command.result)

    def return_book(self):
        title = self.borrow_return_var.get()
        command = ReturnBookCommand(self.library, title)
        command.execute()
        messagebox.showinfo("Result", command.result)

    def add_third_party_books(self):
        third_party_api = ThirdPartyBookAPI()
        third_party_books = third_party_api.get_books()
        for third_party_book in third_party_books:
            adapted_book = BookAdapter(third_party_book)
            self.library.books.append(adapted_book)
            self.notifier.add_new_book(adapted_book)
        messagebox.showinfo("Success", "Third-party books added to the library.")



In [ ]:
client = Client()
client.mainloop()